# Drag-wrench surrogate

Trains the FFNN that replaces the ray-casting drag model inside Isaac Lab.

# 0. Imports and seeds

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit
from sklearn.metrics import r2_score
import os
import re
import time
from glob import glob
from itertools import product

from scipy import spatial

try:
    from adjustText import adjust_text
except ImportError:  # optional, only used for the grid-search plots
    def adjust_text(*a, **k):
        return None

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 1. Load the dataset

One `.pt` per orbital window (episode), with `inputs [N, 22]`, `outputs [N, 6]`,
`diagnostics [N, 26]` and a `metadata` dict.

In [ ]:
dataset_path = "datasets/episodes/"  # folder with the episode_*.pt files

episode_files = sorted(glob(os.path.join(dataset_path, "episode_*.pt")))
print(f"Loading from {dataset_path}")
print(f"Found {len(episode_files)} episode files.")
if not episode_files:
    raise FileNotFoundError(f"no episode_*.pt under {dataset_path}")

all_inputs, all_outputs, all_metadata = [], [], []
for ep_file in episode_files:
    data = torch.load(ep_file, weights_only=False)
    all_inputs.append(data['inputs'])
    all_outputs.append(data['outputs'])
    all_metadata.append(data['metadata'])

inputs = torch.cat(all_inputs, dim=0)
outputs = torch.cat(all_outputs, dim=0)
metadata = all_metadata[0]

# Episode label per sample, used by the grouped hold-out of Section 4
episode_labels = np.concatenate(
    [np.full(len(x), k, dtype=np.int64) for k, x in enumerate(all_inputs)])

print(f"Episodes loaded : {len(all_inputs)}")
print(f"Samples         : {len(inputs)}")
print(f"Input shape     : {tuple(inputs.shape)}")
print(f"Output shape    : {tuple(outputs.shape)}")

# Datasets collected with the same seed share their random attitudes and boom configurations
_seeds = {m.get('seed', None) for m in all_metadata}
print(f"Collection seed(s): {_seeds}")

print("\nMetadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

# 2. Preprocessing: NaN / Inf removal

In [ ]:
print(f"Inputs  contain NaN: {torch.isnan(inputs).any().item()}  "
      f"inf: {torch.isinf(inputs).any().item()}")
print(f"Outputs contain NaN: {torch.isnan(outputs).any().item()}  "
      f"inf: {torch.isinf(outputs).any().item()}")

valid_mask = ((~torch.isnan(inputs).any(dim=1)) & (~torch.isinf(inputs).any(dim=1))
              & (~torch.isnan(outputs).any(dim=1)) & (~torch.isinf(outputs).any(dim=1)))

filtered_inputs = inputs[valid_mask]
filtered_outputs = outputs[valid_mask]
episode_labels = episode_labels[valid_mask.numpy()]

print(f"\nKept {int(valid_mask.sum())} of {len(inputs)} samples "
      f"({int((~valid_mask).sum())} dropped)")
print(f"Episode labels: {len(episode_labels)} "
      f"over {len(np.unique(episode_labels))} episodes")

# 3. Feature scaling

The inputs live in very different units (quaternions, boom extensions, positions in km, velocities in
km/s, Earth's angular rate, air density) and the outputs mix forces and torques. Without scaling, the
MSE is dominated by the component with the largest variance and a single learning rate suits no feature.

| Feature group | Treatment | Reason |
|---|---|---|
| Unit quaternions | **bypassed** (mean 0, scale 1) | already `O(1)`, and an affine map per component breaks `‖q‖ = 1` |
| Near-constant columns (`atm_angular_velocity`) | **centred only** | `std ≈ 0`, so dividing by it amplifies noise |
| Everything else | **z-score** | conditions the input covariance |
| Outputs (force, torque) | **z-score** | otherwise the drag force dominates the loss and the torques are ignored |

Z-score rather than min-max because the dataset deliberately contains tumbling attitudes, and a few extreme
samples would set the min-max scale for all the others.

**Density column.** Drag is linear in ρ, so a linear z-score keeps the map the network has to learn linear
in ρ, whereas `log10` would turn it into an exponential. The counter-argument is conditioning when ρ spans
many decades. `LOG_DENSITY` selects between the two, and the next cells measure the actual spread.

**Metrics.** `R²` is invariant to an affine rescale of the targets. `MAE`, `max_abs_error`, `nrmse` and
`mean_rel_error` are not: they are in units of output-scaler sigmas unless `output_scaler` is passed to
`evaluate_model`, so report the physical ones.

In [ ]:
# 3.1 Feature layout, parsed from the dataset metadata so that the scaler is
# applied to the right columns even if the collection order changes
def parse_feature_layout(spec_list, expected_dim=None, label=""):
    """['name(4)', 'other(3)'] -> [(name, [col indices]), ...]"""
    layout, col = [], 0
    for spec in spec_list:
        m = re.match(r"^\s*([A-Za-z0-9_]+)\s*\((\d+)\)\s*$", spec)
        if m is None:
            raise ValueError(f"cannot parse feature spec {spec!r}")
        name, dim = m.group(1), int(m.group(2))
        layout.append((name, list(range(col, col + dim))))
        col += dim
    if expected_dim is not None and col != expected_dim:
        raise ValueError(f"{label} layout sums to {col} columns but the tensor "
                         f"has {expected_dim}")
    return layout, col


_DEFAULT_INPUT_SPEC = ['spacecraft_orientation(4)', 'boom_extensions(4)',
                       'orbit_orientation(4)', 'orbit_position(3)',
                       'orbit_velocity(3)', 'atm_angular_velocity(3)',
                       'air_density(1)']
_DEFAULT_OUTPUT_SPEC = ['net_force(3)', 'net_torque(3)']

input_spec = metadata.get('input_features', _DEFAULT_INPUT_SPEC)
output_spec = metadata.get('output_features', _DEFAULT_OUTPUT_SPEC)

INPUT_LAYOUT, n_in = parse_feature_layout(input_spec, inputs.shape[1], "input")
OUTPUT_LAYOUT, n_out = parse_feature_layout(output_spec, outputs.shape[1], "output")

INPUT_NAMES, OUTPUT_NAMES = [], []
for name, cols in INPUT_LAYOUT:
    INPUT_NAMES += [f"{name}[{k}]" for k in range(len(cols))]
for name, cols in OUTPUT_LAYOUT:
    OUTPUT_NAMES += [f"{name}[{k}]" for k in range(len(cols))]

QUATERNION_GROUPS = [(n, c) for n, c in INPUT_LAYOUT
                     if len(c) == 4 and re.search(r"orient|quat", n, re.I)]
QUATERNION_COLUMNS = [c for _, cols in QUATERNION_GROUPS for c in cols]
DENSITY_COLUMNS = [c for n, cols in INPUT_LAYOUT
                   if re.search(r"densit", n, re.I) for c in cols]
RHO_COL = DENSITY_COLUMNS[0] if DENSITY_COLUMNS else 21

print("=" * 78)
print("INPUT LAYOUT  (source: metadata['input_features'])"
      if 'input_features' in metadata else
      "INPUT LAYOUT  (source: HARD-CODED FALLBACK - verify against your generator!)")
print("=" * 78)
for name, cols in INPUT_LAYOUT:
    tag = ""
    if cols[0] in QUATERNION_COLUMNS:
        tag = "  <- unit quaternion, BYPASSED by the scaler"
    elif cols[0] in DENSITY_COLUMNS:
        tag = "  <- density, optional log10 (see LOG_DENSITY)"
    print(f"  cols {str(cols):<12} {name}{tag}")
print(f"  total: {n_in} columns (tensor has {inputs.shape[1]})")

print("\nOUTPUT LAYOUT")
for name, cols in OUTPUT_LAYOUT:
    print(f"  cols {str(cols):<12} {name}")

print(f"\nQUATERNION_COLUMNS -> {QUATERNION_COLUMNS}")
print(f"DENSITY_COLUMNS    -> {DENSITY_COLUMNS}  (RHO_COL = {RHO_COL})")

In [ ]:
# 3.2 Magnitudes of the raw features (reporting only, the scaler is fitted on the training split)
_X_diag = filtered_inputs.double()
_y_diag = filtered_outputs.double()


def describe_columns(t, names, title):
    mean = t.mean(0)
    std = t.std(0, unbiased=False)
    amin, amax = t.min(0).values, t.max(0).values
    a = t.abs()
    hi = a.max(0).values
    lo = torch.where(a > 0, a, torch.full_like(a, float('inf'))).min(0).values
    ok = torch.isfinite(lo) & (lo > 0) & (hi > 0)
    decades = torch.where(ok, torch.log10(hi / lo.clamp_min(1e-300)),
                          torch.zeros_like(hi))
    print("=" * 108)
    print(title)
    print("=" * 108)
    print(f"{'#':>3} {'feature':<28} {'min':>12} {'max':>12} {'mean':>12} "
          f"{'std':>12} {'|x| decades':>12}")
    print("-" * 108)
    for i, nm in enumerate(names):
        print(f"{i:>3} {nm:<28} {amin[i]:>12.4e} {amax[i]:>12.4e} "
              f"{mean[i]:>12.4e} {std[i]:>12.4e} {decades[i]:>12.2f}")
    return mean, std


print("Samples used for the diagnostic:", _X_diag.shape[0])
in_mean_raw, in_std_raw = describe_columns(_X_diag, INPUT_NAMES,
                                           "RAW INPUT FEATURES (before scaling)")
out_mean_raw, out_std_raw = describe_columns(_y_diag, OUTPUT_NAMES,
                                             "RAW OUTPUT FEATURES (before scaling)")


def spread_report(std, names, label):
    nz = torch.where(std > 0, std, torch.full_like(std, float('inf')))
    if not torch.isfinite(nz).any():
        print(f"{label}: all columns constant?!")
        return
    hi_i, lo_i = int(torch.argmax(std)), int(torch.argmin(nz))
    ratio = float(std[hi_i] / std[lo_i])
    print(f"\n{label}")
    print(f"  largest  std : {std[hi_i]:.4e}  ({names[hi_i]})")
    print(f"  smallest std : {std[lo_i]:.4e}  ({names[lo_i]})  [non-zero only]")
    print(f"  ratio        : {ratio:.3e}  ->  {np.log10(ratio):.1f} orders")
    print("  => a single learning rate cannot suit all features; z-scoring is justified."
          if ratio > 1e2 else
          "  => spread is mild; scaling buys conditioning, not correctness.")


spread_report(in_std_raw, INPUT_NAMES, "INPUT std spread")
spread_report(out_std_raw, OUTPUT_NAMES, "OUTPUT std spread")

const_cols = [i for i in range(len(INPUT_NAMES))
              if float(in_std_raw[i]) <= 1e-8 * max(abs(float(in_mean_raw[i])), 1e-300)]
print("\nNEAR-CONSTANT INPUT COLUMNS (std/|mean| <= 1e-8):")
for i in const_cols:
    print(f"  col {i:>2} {INPUT_NAMES[i]:<28} value = {in_mean_raw[i]:.6e}")
if not const_cols:
    print("  none")

print("\n" + "=" * 78)
print("DYNAMIC RANGE THE NETWORK MUST LEARN")
print("=" * 78)
_pos = _X_diag[:, RHO_COL][_X_diag[:, RHO_COL] > 0]
_dec_rho = float(torch.log10(_pos.max() / _pos.min()))
print(f"air density spans {_dec_rho:.2f} decades: "
      f"[{_pos.min():.3e}, {_pos.max():.3e}] kg/m^3")
print("Drag is linear in rho, so the wrench inherits that span")
for i, nm in enumerate(OUTPUT_NAMES):
    a = _y_diag[:, i].abs()
    a = a[a > 0]
    print(f"  {nm:<18} {float(torch.log10(a.max()/a.min())):>6.2f} decades")

# False: linear z-score, keeps rho -> wrench linear (preferred if rho spans < ~2 decades)
# True: log10 then z-score, better conditioned over many decades but the network must learn an exponential
LOG_DENSITY = False
LOG_COLUMNS = DENSITY_COLUMNS if LOG_DENSITY else []
print(f"\nLOG_DENSITY = {LOG_DENSITY}  ->  LOG_COLUMNS = {LOG_COLUMNS}")
if _dec_rho > 2.0 and not LOG_DENSITY:
    print(f"  NOTE: rho spans {_dec_rho:.1f} decades, also try LOG_DENSITY = True")

In [ ]:
# 3.3 The scaler, used for training, testing and RL inference
class FeatureScaler:
    """Elementwise standardisation  x_hat = (f(x) - mean) / scale

    f is the identity except on `log_columns`, where it is log10.

    - `skip_columns`: mean = 0, scale = 1 (pass-through), used for the unit quaternions
    - near-constant columns: scale forced to 1, so they are only centred
    - statistics are accumulated in float64 (the air density variance can be ~1e-24) and stored in float32
    - `state_dict()` is a plain dict of tensors, so inference only needs torch
    """

    def __init__(self, skip_columns=None, log_columns=None, name="scaler",
                 const_rel_tol=1e-8, const_abs_tol=1e-30, log_floor=1e-38):
        self.name = name
        self.skip_columns = sorted({int(c) for c in (skip_columns or [])})
        self.log_columns = sorted({int(c) for c in (log_columns or [])})
        clash = set(self.skip_columns) & set(self.log_columns)
        if clash:
            raise ValueError(f"columns {sorted(clash)} are both skipped and log-scaled")
        self.const_rel_tol = float(const_rel_tol)
        self.const_abs_tol = float(const_abs_tol)
        self.log_floor = float(log_floor)
        self.mean = None
        self.scale = None
        self.raw_std = None
        self.constant_columns = []
        self.n_fit = 0
        self.fitted = False

    @staticmethod
    def _as_tensor(x):
        return torch.from_numpy(x) if isinstance(x, np.ndarray) else x

    def _check(self):
        if not self.fitted:
            raise RuntimeError(f"{self.name}: fit() must be called first")

    def _log_forward(self, t):
        if not self.log_columns:
            return t
        idx = torch.as_tensor(self.log_columns, device=t.device, dtype=torch.long)
        t = t.clone()
        t[..., idx] = torch.log10(t[..., idx].clamp_min(self.log_floor))
        return t

    def _log_inverse(self, t):
        if not self.log_columns:
            return t
        idx = torch.as_tensor(self.log_columns, device=t.device, dtype=torch.long)
        t = t.clone()
        t[..., idx] = torch.pow(torch.tensor(10.0, dtype=t.dtype, device=t.device),
                                t[..., idx])
        return t

    def fit(self, X):
        t = self._as_tensor(X)
        if t.dim() != 2:
            raise ValueError("expected a 2-D [N, D] tensor")
        t = t.double()
        tl = self._log_forward(t)

        mean = tl.mean(dim=0)
        std = tl.std(dim=0, unbiased=False)
        self.raw_std = std.clone().float()

        near_const = ((std <= self.const_rel_tol * mean.abs().clamp_min(1e-300))
                      | (std <= self.const_abs_tol))
        scale = torch.where(near_const, torch.ones_like(std), std)
        self.constant_columns = torch.nonzero(near_const).flatten().tolist()

        if self.skip_columns:
            sk = torch.as_tensor(self.skip_columns, dtype=torch.long)
            mean[sk] = 0.0
            scale[sk] = 1.0

        self.mean = mean.float()
        self.scale = scale.float()
        self.n_fit = int(t.shape[0])
        self.fitted = True
        return self

    def transform(self, x):
        self._check()
        is_np = isinstance(x, np.ndarray)
        t = self._as_tensor(x)
        dtype = t.dtype if t.is_floating_point() else torch.float32
        t = self._log_forward(t.to(dtype))
        m = self.mean.to(device=t.device, dtype=dtype)
        s = self.scale.to(device=t.device, dtype=dtype)
        out = (t - m) / s
        return out.cpu().numpy() if is_np else out

    def inverse_transform(self, x):
        self._check()
        is_np = isinstance(x, np.ndarray)
        t = self._as_tensor(x)
        dtype = t.dtype if t.is_floating_point() else torch.float32
        t = t.to(dtype)
        m = self.mean.to(device=t.device, dtype=dtype)
        s = self.scale.to(device=t.device, dtype=dtype)
        out = self._log_inverse(t * s + m)
        return out.cpu().numpy() if is_np else out

    def state_dict(self):
        self._check()
        return {
            "name": self.name,
            "mean": self.mean.clone(),
            "scale": self.scale.clone(),
            "raw_std": self.raw_std.clone(),
            "skip_columns": list(self.skip_columns),
            "log_columns": list(self.log_columns),
            "constant_columns": list(self.constant_columns),
            "n_fit": self.n_fit,
        }

    @classmethod
    def from_state_dict(cls, sd):
        obj = cls(skip_columns=sd.get("skip_columns"),
                  log_columns=sd.get("log_columns"),
                  name=sd.get("name", "scaler"))
        obj.mean = sd["mean"].clone()
        obj.scale = sd["scale"].clone()
        obj.raw_std = sd.get("raw_std", torch.ones_like(obj.scale)).clone()
        obj.constant_columns = list(sd.get("constant_columns", []))
        obj.n_fit = int(sd.get("n_fit", 0))
        obj.fitted = True
        return obj

    def summary(self, names=None, title=None):
        self._check()
        names = names or [f"col[{i}]" for i in range(len(self.mean))]
        print("=" * 96)
        print(title or f"{self.name}: constants fitted on {self.n_fit} samples")
        print("=" * 96)
        print(f"{'#':>3} {'feature':<28} {'treatment':<14} {'mean':>14} "
              f"{'scale':>14} {'raw std':>14}")
        print("-" * 96)
        for i, nm in enumerate(names):
            if i in self.skip_columns:
                how = "bypass"
            elif i in self.constant_columns:
                how = "centre only"
            elif i in self.log_columns:
                how = "log10+zscore"
            else:
                how = "z-score"
            print(f"{i:>3} {nm:<28} {how:<14} {self.mean[i]:>14.6e} "
                  f"{self.scale[i]:>14.6e} {self.raw_std[i]:>14.6e}")
        print("-" * 96)
        return self

    def __repr__(self):
        st = "fitted" if self.fitted else "unfitted"
        return (f"<FeatureScaler {self.name} [{st}] n_fit={self.n_fit} "
                f"skip={self.skip_columns} log={self.log_columns} "
                f"const={self.constant_columns}>")

# 4. Split the dataset

`SPLIT_BY_GROUP` selects what the test set measures:

* **`False`: `StratifiedShuffleSplit` on the episode.** Every orbital window is in train, validation and test,
  so the test set measures the interpolation error at new points from the same generator. This is the split
  used for the deployed model.
* **`True`: `GroupShuffleSplit` on the episode.** Whole windows are withheld, so it measures the
  generalisation to unseen orbital conditions. Run it once as a diagnostic.

In [ ]:
SPLIT_BY_GROUP = False

X = filtered_inputs.numpy()
y = filtered_outputs.numpy()
episode_labels = np.asarray(episode_labels)

if SPLIT_BY_GROUP:
    splitter1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    idx1 = splitter1.split(X, y, groups=episode_labels)
else:
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    idx1 = splitter1.split(X, episode_labels)

for train_index, temp_index in idx1:
    X_train, y_train = X[train_index], y[train_index]
    X_temp, y_temp = X[temp_index], y[temp_index]
    episode_labels_temp = episode_labels[temp_index]

if SPLIT_BY_GROUP:
    splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
    idx2 = splitter2.split(X_temp, y_temp, groups=episode_labels_temp)
else:
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
    idx2 = splitter2.split(X_temp, episode_labels_temp)

for val_index, test_index in idx2:
    X_val, y_val = X_temp[val_index], y_temp[val_index]
    X_test, y_test = X_temp[test_index], y_temp[test_index]

# Absolute indices into X
val_index_abs = temp_index[val_index]
test_index_abs = temp_index[test_index]

# Raw (unscaled) tensors, for the deployment check and the evaluation sections
X_train_raw = torch.tensor(X_train, dtype=torch.float32)
y_train_raw = torch.tensor(y_train, dtype=torch.float32)
X_val_raw = torch.tensor(X_val, dtype=torch.float32)
y_val_raw = torch.tensor(y_val, dtype=torch.float32)
X_test_raw = torch.tensor(X_test, dtype=torch.float32)
y_test_raw = torch.tensor(y_test, dtype=torch.float32)

# Fit the scalers on the training split only, to avoid leaking validation/test statistics
input_scaler = FeatureScaler(skip_columns=QUATERNION_COLUMNS,
                             log_columns=LOG_COLUMNS,
                             name="input_scaler").fit(X_train_raw)
output_scaler = FeatureScaler(name="output_scaler").fit(y_train_raw)

X_train = input_scaler.transform(X_train_raw)
X_val = input_scaler.transform(X_val_raw)
X_test = input_scaler.transform(X_test_raw)
y_train = output_scaler.transform(y_train_raw)
y_val = output_scaler.transform(y_val_raw)
y_test = output_scaler.transform(y_test_raw)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)


# Persist the scalers so that the same preprocessing can be reproduced inside Isaac Lab
scalers_path = os.path.join(dataset_path, "feature_scalers_noq.pt")
torch.save({
    "input_scaler": input_scaler.state_dict(),
    "output_scaler": output_scaler.state_dict(),
    "input_features": input_spec,
    "output_features": output_spec,
    "fitted_on": "training split only",
    "n_train": int(X_train.shape[0]),
    "dataset_path": dataset_path,
    "split_by_group": SPLIT_BY_GROUP,
    "log_density": LOG_DENSITY,
    "parameterisation": "direct wrench (no dynamic-pressure reduction)",
}, scalers_path)
print(f"Saved fitted scalers to {scalers_path}")
print(input_scaler)
print(output_scaler)

print(f"\nFinal splits  (SPLIT_BY_GROUP = {SPLIT_BY_GROUP}):")
print(f"  Total      : {len(X)}")
print(f"  Train      : {len(X_train)} ({len(X_train)/len(X):.2%})")
print(f"  Validation : {len(X_val)} ({len(X_val)/len(X):.2%})")
print(f"  Test       : {len(X_test)} ({len(X_test)/len(X):.2%})")

# What the split actually withheld
_e_tr = set(np.unique(episode_labels[train_index]).tolist())
_e_va = set(np.unique(episode_labels[val_index_abs]).tolist())
_e_te = set(np.unique(episode_labels[test_index_abs]).tolist())
print(f"\n  episodes in train/val/test : {len(_e_tr)} / {len(_e_va)} / {len(_e_te)}")
print(f"  test episodes unseen in train: {len(_e_te - _e_tr)} of {len(_e_te)}")
print("  -> " + ("GROUPED hold-out: generalisation to unseen orbital windows"
                 if _e_te - _e_tr else
                 "STRATIFIED split: interpolation at unseen samples, not generalisation across orbital conditions"))

## 4b. Scaling constants and verification

In [ ]:
input_scaler.summary(INPUT_NAMES,
                     title="INPUT SCALER - fitted on the TRAINING split only")
print()
output_scaler.summary(OUTPUT_NAMES,
                      title="OUTPUT SCALER - fitted on the TRAINING split only")


def _spread(t):
    s = t.std(0, unbiased=False)
    s = s[s > 0]
    return float(s.max() / s.min()) if s.numel() else float('nan')


print("\nCONDITIONING CHECK (ratio max/min of per-column std, non-zero columns)")
print(f"  inputs  : raw {_spread(X_train_raw.double()):.3e}"
      f"  ->  scaled {_spread(X_train.double()):.3e}")
print(f"  outputs : raw {_spread(y_train_raw.double()):.3e}"
      f"  ->  scaled {_spread(y_train.double()):.3e}")

_zs = [i for i in range(X_train.shape[1])
       if i not in input_scaler.skip_columns and i not in input_scaler.constant_columns]
_zs = _zs or list(range(X_train.shape[1]))
print(f"\nScaled TRAIN inputs on z-scored cols: mean = {X_train[:, _zs].mean():.3e}, "
      f"std = {X_train[:, _zs].std():.6f}  (expected 0 and 1)")
print(f"Scaled VAL   inputs on z-scored cols: mean = {X_val[:, _zs].mean():.3e}, "
      f"std = {X_val[:, _zs].std():.6f}  (close, NOT exact -> no leakage)")
print(f"Scaled TRAIN outputs: mean = {y_train.mean():.3e}, std = {y_train.std():.6f}")

# Density-shift sensitivity: a density outside the fitted range moves both the scaled input and the target
_rho_raw = X_train_raw[:, RHO_COL].double()
_rho_pos = _rho_raw[_rho_raw > 0]
print("\nDENSITY-SHIFT SENSITIVITY")
print(f"  training rho : [{_rho_pos.min():.4e}, {_rho_pos.max():.4e}] kg/m^3 "
      f"({float(torch.log10(_rho_pos.max()/_rho_pos.min())):.2f} decades)")
print(f"  scaled col   : mean {X_train[:, RHO_COL].double().mean():+.3f}, "
      f"std {X_train[:, RHO_COL].double().std():.3f}")
if RHO_COL in input_scaler.log_columns:
    print(f"  a 10x density change -> "
          f"{1.0/max(float(input_scaler.scale[RHO_COL]), 1e-12):.2f} sigma "
          "in the scaled input (log10 column)")
else:
    _sig = float(_rho_pos.median() * 9 / max(float(input_scaler.scale[RHO_COL]), 1e-300))
    print(f"  a 10x density change -> ~{_sig:.1f} sigma in the scaled input "
          "(LINEAR column)")
print("  effect on the target: proportional, so the network must extrapolate on both sides")

In [ ]:
# Round-trip check and persistence
_probe = X_train_raw[:4096]
_rt = input_scaler.inverse_transform(input_scaler.transform(_probe))
_abs = (_rt - _probe).abs()
# Error relative to the column spread: a plain relative error blows up near zero (float32 cancellation)
_colstd = _probe.std(0, unbiased=False).clamp_min(1e-30)
print("ROUND-TRIP inverse_transform(transform(x)) on 4096 raw input samples")
print(f"  max abs error              : {_abs.max():.3e}")
print(f"  max abs error / column std : {(_abs / _colstd).max():.3e}   "
      "(float32 eps = 1.19e-07)")
_worst = int(torch.argmax((_abs / _colstd).max(0).values))
print(f"  worst column               : {_worst} ({INPUT_NAMES[_worst]})")

if QUATERNION_COLUMNS:
    _qc = torch.as_tensor(QUATERNION_COLUMNS, dtype=torch.long)
    _qs = input_scaler.transform(_probe)[:, _qc]
    print(f"  quaternion cols: max |scaled - raw| = "
          f"{(_qs - _probe[:, _qc]).abs().max():.3e}  (must be 0, bypassed)")

_probe_y = y_train_raw[:4096]
_rt_y = output_scaler.inverse_transform(output_scaler.transform(_probe_y))
_den = _probe_y.abs().clamp_min(_probe_y.abs().median().clamp_min(1e-30))
print(f"  outputs max abs error : {(_rt_y - _probe_y).abs().max():.3e}")
print(f"  outputs max REL error : {((_rt_y - _probe_y).abs() / _den).max():.3e}")

# Reload from disk and check parity with the in-memory scalers
_sd = torch.load(scalers_path, weights_only=False)
_in_rt = FeatureScaler.from_state_dict(_sd["input_scaler"])
_out_rt = FeatureScaler.from_state_dict(_sd["output_scaler"])
assert torch.equal(_in_rt.transform(_probe), input_scaler.transform(_probe)), \
    "reloaded input scaler does not match the fitted one"
assert torch.equal(_out_rt.inverse_transform(output_scaler.transform(_probe_y)),
                   output_scaler.inverse_transform(output_scaler.transform(_probe_y))), \
    "reloaded output scaler does not match the fitted one"
print(f"\nReloaded {scalers_path} -> transforms match exactly.")

print("\nUse in Isaac Lab: load drag_surrogate_deploy_noq.pt (scalers included), or apply "
      "(x - mean) / scale around the network.")
print("Units: position in km and velocity in km/s, as in the training data. SI units will not raise "
      "an error but give a wrong scaled input.")

# 5. Model

In [ ]:
class DragForceNN(nn.Module):
    """Fully-connected ReLU network (22 features in, 6 wrench components out)

    `use_norm` inserts a LayerNorm after every hidden Linear
    """

    def __init__(self, input_size, hidden_size, num_layers, output_size,
                 use_norm=False):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList() if use_norm else None

        self.layers.append(nn.Linear(input_size, hidden_size))
        if use_norm:
            self.norms.append(nn.LayerNorm(hidden_size))

        for _ in range(num_layers - 1):
            self.layers.append(nn.Linear(hidden_size, hidden_size))
            if use_norm:
                self.norms.append(nn.LayerNorm(hidden_size))

        self.output_layer = nn.Linear(hidden_size, output_size)
        self.activation = nn.ReLU()

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if self.norms is not None:
                x = self.norms[i](x)
            x = self.activation(x)
        return self.output_layer(x)

# 6. Training and evaluation

In [ ]:
def plot_training_curve(train_losses, val_losses, save_dir, name):
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (scaled units)')
    plt.title('Training and validation loss - ' + name)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(os.path.join(save_dir, f"loss_{name}.png"), dpi=130)


def train_model(model, train_loader, val_loader, initial_lr=0.001, device=device,
                patience=10, max_epochs=1000, scheduler_patience=5, verbose=True):
    """Adam + ReduceLROnPlateau + early stopping on the validation MSE (loaders yield scaled tensors)"""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=initial_lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                  patience=scheduler_patience)

    best_val_loss = float('inf')
    best_model_state = None
    no_improve_count = 0
    train_losses, val_losses = [], []

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += criterion(model(xb), yb).item() * xb.size(0)
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)

        scheduler.step(val_loss)

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f'Epoch {epoch+1}/{max_epochs}, Train {train_loss:.6f}, '
                  f'Val {val_loss:.6f}, LR {optimizer.param_groups[0]["lr"]:.6f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = {k: v.detach().clone()
                                for k, v in model.state_dict().items()}
            no_improve_count = 0
        else:
            no_improve_count += 1
        if no_improve_count >= patience:
            if verbose:
                print(f'Early stopping at epoch {epoch+1}')
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return {'model': model,
            'train_loss': train_losses[-1],
            'val_loss': best_val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'epochs': len(train_losses)}

In [ ]:
# Metrics on the scaled tensors are dimensionless (output-scaler sigmas). Only R2 is unaffected by the
# scaling: pass `output_scaler` to evaluate_model to get the physical-unit numbers as well
def _metric_block(T, P, names, label, rel_floor_frac=1e-2, verbose=True):
    """Metrics for one representation (scaled or physical) of the same data"""
    err = P - T
    n_out = T.shape[1]

    mse = float((err ** 2).mean())
    rmse = float(np.sqrt(mse))
    mae = float(np.abs(err).mean())
    max_abs_error = float(np.abs(err).max())

    nrmse = float(rmse / (np.abs(T).max() + 1e-12))
    nrmse_std = float(rmse / (T.std() + 1e-30))

    # The mean relative error explodes where the components cross zero: restrict it to the samples
    # whose target is above a fraction of the component std, and report the coverage
    rel_vals, kept, total = [], 0, 0
    for i in range(n_out):
        floor = rel_floor_frac * T[:, i].std()
        m = np.abs(T[:, i]) > floor
        total += m.size
        kept += int(m.sum())
        if m.any():
            rel_vals.append(np.abs(err[m, i] / T[m, i]))
    mean_rel_error = float(np.concatenate(rel_vals).mean()) if rel_vals else float('nan')
    rel_coverage = kept / max(total, 1)

    # Constancy is tested relative to the magnitude (np.allclose would flag targets of ~1e-10 N as constant)
    def _is_constant(v):
        sd = float(v.std())
        return (not np.isfinite(sd)) or sd <= 1e-30 or sd <= 1e-12 * abs(float(v.mean()))

    comp_r2 = [0.0 if _is_constant(T[:, i]) else float(r2_score(T[:, i], P[:, i]))
               for i in range(n_out)]
    if all(_is_constant(T[:, i]) for i in range(n_out)):
        print("Warning: all target columns are constant, R2 is not meaningful")
        r2 = 0.0
    else:
        r2 = float(np.mean(comp_r2))

    comp = {'mae': np.abs(err).mean(0),
            'rmse': np.sqrt((err ** 2).mean(0)),
            'max_abs_error': np.abs(err).max(0),
            'r2': np.array(comp_r2),
            'target_std': T.std(0)}

    if verbose:
        print(f"\n--- {label} ---")
        print(f"{'component':<20} {'MAE':>12} {'RMSE':>12} {'max|err|':>12} "
              f"{'R2':>9} {'std(target)':>13} {'RMSE/std':>10}")
        print("-" * 94)
        for i, nm in enumerate(names):
            print(f"{nm:<20} {comp['mae'][i]:>12.4e} {comp['rmse'][i]:>12.4e} "
                  f"{comp['max_abs_error'][i]:>12.4e} {comp['r2'][i]:>9.5f} "
                  f"{comp['target_std'][i]:>13.4e} "
                  f"{comp['rmse'][i] / (comp['target_std'][i] + 1e-30):>10.4f}")
        print("-" * 94)
        print(f"{'AGGREGATE':<20} {mae:>12.4e} {rmse:>12.4e} {max_abs_error:>12.4e} "
              f"{r2:>9.5f} {T.std():>13.4e} {nrmse_std:>10.4f}")
        print(f"mean relative error: {mean_rel_error:.4f} "
              f"(over {100*rel_coverage:.1f}% of entries above the floor)")

    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'max_abs_error': max_abs_error,
            'nrmse': nrmse, 'nrmse_std': nrmse_std, 'r2_score': r2,
            'mean_rel_error': mean_rel_error, 'rel_coverage': rel_coverage,
            'per_component': comp}


def evaluate_model(model, test_loader, device=device, output_scaler=None,
                   names=None, verbose=True):
    """Scaled and (optionally) physical metrics of `model` on `test_loader`, plus the single-sample latency"""
    names = names or OUTPUT_NAMES
    model.eval()
    P, T = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            P.append(model(xb.to(device)).cpu().numpy())
            T.append(yb.numpy())
    P = np.vstack(P)
    T = np.vstack(T)

    out = _metric_block(T, P, names, "SCALED units (output-scaler sigmas)",
                        verbose=verbose)
    out['n_samples'] = int(T.shape[0])

    if output_scaler is not None:
        Tp = output_scaler.inverse_transform(torch.tensor(T)).numpy()
        Pp = output_scaler.inverse_transform(torch.tensor(P)).numpy()
        phys = _metric_block(Tp, Pp, names, "PHYSICAL units (N, N*m)",
                             verbose=verbose)
        out['physical'] = phys
        out['targets_physical'] = Tp
        out['predictions_physical'] = Pp

    # Inference latency, one sample
    x1 = next(iter(test_loader))[0][:1].to(device)
    with torch.no_grad():
        for _ in range(10):
            model(x1)
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(100):
            model(x1)
        if device == 'cuda':
            torch.cuda.synchronize()
        out['inference_time_ms'] = (time.perf_counter() - t0) / 100 * 1000

    out['parameters'] = sum(p.numel() for p in model.parameters())
    out['targets'] = T
    out['predictions'] = P
    if verbose:
        print(f"\nparameters {out['parameters']}, "
              f"inference {out['inference_time_ms']:.4f} ms/sample")
    return out

# 7. Hyperparameter grid search

Every model is trained on the same split with the same scalers and compared on the **test** set, which is
used neither for training nor for early stopping.

With the direct parameterisation the target variance is dominated by the high-density part of the orbit, so
a high aggregate `R²` mostly certifies the accuracy where the drag is strongest. Section 8 breaks the error
down by density.

In [ ]:
param_grid = {
    'num_layers': [1, 3],
    'hidden_size': [32, 64, 128],
    'batch_size': [64, 128, 256],
    'use_norm': [True, False]
}

results_dir = os.path.join(dataset_path, "grid_search_results_noq")
os.makedirs(results_dir, exist_ok=True)

combos = list(product(param_grid['num_layers'], param_grid['hidden_size'],
                      param_grid['batch_size'], param_grid['use_norm']))
print(f"{len(combos)} configurations -> {results_dir}")

In [ ]:
results = []
t_start = time.time()

for k, (num_layers, hidden_size, batch_size, use_norm) in enumerate(combos):
    name = f"layers{num_layers}_hidden{hidden_size}_batch{batch_size}_norm{use_norm}"
    print(f"\n[{k+1}/{len(combos)}] {name}")

    torch.manual_seed(seed)  # same initialisation for every configuration
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = DragForceNN(input_size=X_train.shape[1], hidden_size=hidden_size,
                        num_layers=num_layers, output_size=y_train.shape[1],
                        use_norm=use_norm).to(device)

    tr = train_model(model, train_loader, val_loader, initial_lr=1e-3,
                     device=device, verbose=False)
    ev = evaluate_model(model, test_loader, device=device,
                        output_scaler=output_scaler, verbose=False)

    torch.save({'model_state_dict': model.state_dict(),
                'hyperparameters': dict(num_layers=num_layers,
                                        hidden_size=hidden_size,
                                        batch_size=batch_size,
                                        use_norm=use_norm),
                'parameterisation': 'direct wrench (no q)'},
               os.path.join(results_dir, f"{name}.pt"))
    plot_training_curve(tr['train_losses'], tr['val_losses'], results_dir, name)
    plt.close('all')

    results.append({
        'name': name,
        'num_layers': num_layers, 'hidden_size': hidden_size,
        'batch_size': batch_size, 'use_norm': use_norm,
        'parameters': ev['parameters'],
        'epochs': tr['epochs'],
        'train_loss': tr['train_loss'], 'val_loss': tr['val_loss'],
        'r2_score': ev['r2_score'],
        'mae': ev['physical']['mae'],
        'rmse': ev['physical']['rmse'],
        'max_abs_error': ev['physical']['max_abs_error'],
        'nrmse': ev['physical']['nrmse'],
        'mean_rel_error': ev['physical']['mean_rel_error'],
        'inference_time_ms': ev['inference_time_ms'],
    })
    print(f"    R2 {ev['r2_score']:.5f}  val {tr['val_loss']:.3e}  "
          f"MAE {ev['physical']['mae']:.3e} N  "
          f"{ev['inference_time_ms']:.4f} ms  {tr['epochs']} epochs")

results_df = pd.DataFrame(results)
results_csv = os.path.join(results_dir, "grid_search_results.csv")
results_df.to_csv(results_csv, index=False)
print(f"\nDone in {(time.time()-t_start)/60:.1f} min -> {results_csv}")
results_df.sort_values('r2_score', ascending=False).head(10)

In [ ]:
def analyze_results(results_df, results_dir):
    """Plots of the grid search (accuracy against error, size and latency) and a size/speed efficiency ranking"""
    os.makedirs(results_dir, exist_ok=True)

    best_model = results_df.loc[results_df['r2_score'].idxmax()]
    print("\n===== Best Model =====")
    for k, v in best_model.items():
        print(f"{k}: {v}")
    with open(os.path.join(results_dir, 'best_model_summary.txt'), 'w') as f:
        for k, v in best_model.items():
            f.write(f"{k}: {v}\n")

    labels = [f"{row['num_layers']}L-{row['hidden_size']}H-{row['batch_size']}B-{'T' if row['use_norm'] else 'F'}N"
              for _, row in results_df.iterrows()]

    def scatter(x, y, xlabel, ylabel, title, filename):
        plt.figure(figsize=(10, 6))
        plt.scatter(results_df[x], results_df[y], alpha=0.7)
        texts = [plt.text(r[x], r[y], labels[i], fontsize=7) for i, (_, r) in enumerate(results_df.iterrows())]
        adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray'))
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.title(title)
        plt.grid(True)
        plt.savefig(os.path.join(results_dir, filename))

    for metric, label in [("mae", "Mean Absolute Error"), ("max_abs_error", "Max Absolute Error"),
                          ("nrmse", "Normalized RMSE"), ("mean_rel_error", "Mean Relative Error")]:
        if metric in results_df.columns:
            scatter('r2_score', metric, "R² Score", label, f"R² vs {label}", f"r2_vs_{metric}.png")

    scatter('val_loss', 'r2_score', "Validation Loss", "R² Score", "Validation Loss vs R² Score", "val_loss_vs_r2.png")
    scatter('parameters', 'r2_score', "Number of Parameters", "R² Score", "Model Size vs Performance",
            "parameters_vs_r2.png")
    scatter('parameters', 'inference_time_ms', "Number of Parameters", "Inference Time (ms)",
            "Model Size vs Inference Time", "parameters_vs_inference_time.png")

    for var in ['hidden_size', 'num_layers', 'use_norm']:
        plt.figure(figsize=(10, 6))
        if results_df[var].nunique() <= 10:
            sns.violinplot(x=var, y='r2_score', data=results_df)
        else:
            sns.boxplot(x=var, y='r2_score', data=results_df)
        plt.title(f'{var.replace("_", " ").title()} vs. R² Score')
        plt.grid(True)
        plt.savefig(os.path.join(results_dir, f'{var}_vs_r2.png'))

    # Pareto frontier of accuracy against inference time
    points = results_df[['r2_score', 'inference_time_ms']].values
    if len(points) > 2:
        hull = spatial.ConvexHull(points)
        plt.figure(figsize=(10, 6))
        plt.scatter(points[:, 0], points[:, 1], alpha=0.6)
        texts = [plt.text(x, y, labels[i], fontsize=7) for i, (x, y) in enumerate(points)]
        adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray'))
        for simplex in hull.simplices:
            plt.plot(points[simplex, 0], points[simplex, 1], 'k--', alpha=0.6)
        plt.xlabel('R² Score')
        plt.ylabel('Inference Time (ms)')
        plt.title('Performance against Inference Time')
        plt.grid(True)
        plt.savefig(os.path.join(results_dir, 'pareto_frontier.png'))

    # Efficiency: weighted sum of the normalised accuracy and speed
    max_r2, min_r2 = results_df['r2_score'].max(), results_df['r2_score'].min()
    max_time, min_time = results_df['inference_time_ms'].max(), results_df['inference_time_ms'].min()
    results_df['norm_r2'] = (results_df['r2_score'] - min_r2) / (max_r2 - min_r2) if max_r2 > min_r2 else 0.5
    results_df['norm_time'] = (1.0 - (results_df['inference_time_ms'] - min_time) / (max_time - min_time)
                               if max_time > min_time else 0.5)

    accuracy_weight, speed_weight = 0.9, 0.1
    results_df['efficiency'] = accuracy_weight * results_df['norm_r2'] + speed_weight * results_df['norm_time']

    sorted_indices = results_df['efficiency'].argsort()[::-1].to_numpy()
    efficiency_sorted = results_df['efficiency'].iloc[sorted_indices]
    labels_sorted = [labels[i] for i in sorted_indices]

    plt.figure(figsize=(12, 6))
    plt.bar(range(len(efficiency_sorted)), efficiency_sorted, color=plt.cm.viridis(efficiency_sorted))
    plt.xticks(range(len(efficiency_sorted)), labels_sorted, rotation=45, ha='right')
    plt.xlabel("Model Configuration")
    plt.ylabel("Efficiency Score")
    plt.title(f"Model Efficiency (Accuracy {accuracy_weight:.1f}, Speed {speed_weight:.1f})")
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "model_efficiency.png"))

    results_df.to_csv(os.path.join(results_dir, 'analyzed_results.csv'), index=False)
    return results_df

In [ ]:
results_df = analyze_results(results_df, results_dir)
best_model_row = results_df.loc[results_df['r2_score'].idxmax()]

# 8. Selected model: evaluation, error vs density, plots

In [ ]:
# Reload the best model and evaluate it in full
SELECTED = str(best_model_row['name'])
_cfg = results_df.loc[results_df['name'] == SELECTED].iloc[0]
print(f"Selected: {SELECTED}")

model = DragForceNN(input_size=X_train.shape[1],
                    hidden_size=int(_cfg['hidden_size']),
                    num_layers=int(_cfg['num_layers']),
                    output_size=y_train.shape[1],
                    use_norm=bool(_cfg['use_norm'])).to(device)
model.load_state_dict(torch.load(os.path.join(results_dir, f"{SELECTED}.pt"),
                                 map_location=device)['model_state_dict'])
model.eval()

test_loader = DataLoader(test_dataset, batch_size=int(_cfg['batch_size']),
                         shuffle=False)
final_metrics = evaluate_model(model, test_loader, device=device,
                               output_scaler=output_scaler, verbose=True)

In [ ]:
# Error against the air density. The aggregate R2 is dominated by the high-density samples,
# so report this decile breakdown alongside it
Tp = final_metrics['targets_physical']
Pp = final_metrics['predictions_physical']
rho_test = X_test_raw[:, RHO_COL].numpy()

_Fm = lambda A: np.linalg.norm(A[:, :3], axis=1)
edges = np.quantile(rho_test, np.linspace(0, 1, 11))

print("MEDIAN RELATIVE ERROR OF |F| BY AIR-DENSITY DECILE")
print(f"{'decile':>7} {'rho median':>14} {'n':>7} {'med |F| [N]':>14} "
      f"{'med rel err':>13} {'p90 rel err':>13}")
print("-" * 74)
for k in range(10):
    m = (rho_test >= edges[k]) & (rho_test <= edges[k + 1])
    if m.sum() < 10:
        continue
    t = np.maximum(_Fm(Tp[m]), 1e-30)
    rel = np.abs(_Fm(Pp[m]) - t) / t
    print(f"{k+1:>7} {np.median(rho_test[m]):>14.3e} {int(m.sum()):>7} "
          f"{np.median(t):>14.3e} {np.median(rel):>13.3e} "
          f"{np.percentile(rel, 90):>13.3e}")
print("-" * 74)
print("The relative error is expected to rise toward the low deciles: the MSE weights each sample by its absolute wrench")

plt.figure(figsize=(9, 5))
_c = np.digitize(rho_test, edges[1:-1])
_med = [np.median(np.abs(_Fm(Pp[_c == k]) - np.maximum(_Fm(Tp[_c == k]), 1e-30))
                  / np.maximum(_Fm(Tp[_c == k]), 1e-30))
        for k in range(10) if (_c == k).sum() > 10]
plt.semilogy(range(1, len(_med) + 1), _med, 'o-')
plt.xlabel('air-density decile (1 = lowest)')
plt.ylabel('median relative error of |F|')
plt.title('Surrogate accuracy across the density range')
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "relerr_vs_density.png"), dpi=130)
plt.show()

In [ ]:
def collect_predictions(model, dataset, output_scaler=None, device=device,
                        batch_size=1024):
    """Run the model over a (scaled) dataset and return (targets, predictions) as arrays

    With `output_scaler` both are de-scaled to N and N*m, otherwise they stay in scaler sigmas
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    preds, targs = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb.to(device)).cpu())
            targs.append(yb.cpu())
    P = torch.cat(preds, 0)
    T = torch.cat(targs, 0)
    if output_scaler is not None:
        P = output_scaler.inverse_transform(P)
        T = output_scaler.inverse_transform(T)
    return T.numpy(), P.numpy()


def _unit_of(name):
    n = name.lower()
    return 'N\u00b7m' if 'torque' in n else ('N' if 'force' in n else '-')


def plot_predictions_vs_targets(T, P, names=None, save_dir=None, tag='val',
                                max_points=2000, sorted_view=True,
                                index_view=True, parity=True, residuals=True,
                                nrows=2, ncols=3, dpi=130):
    """Ground truth vs prediction for every component: sorted view, dataset-order view, parity plot and
    residual histogram (`max_points=None` plots every sample)"""
    names = names or OUTPUT_NAMES
    n_out, n_samples = T.shape[1], T.shape[0]
    figs = {}

    def _grid(title):
        fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 3.6 * nrows),
                                 dpi=dpi)
        axes = np.atleast_1d(axes).ravel()
        fig.suptitle(title, fontsize=13)
        return fig, axes

    def _finish(fig, key):
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            path = os.path.join(save_dir, f"{tag}_{key}.png")
            fig.savefig(path, bbox_inches='tight')
            print(f"  saved {path}")
        figs[key] = fig

    if sorted_view:
        fig, axes = _grid(f"Truth vs prediction - {tag}, sorted by true value "
                          f"({n_samples} samples)")
        for i in range(n_out):
            ax = axes[i]
            order = np.argsort(T[:, i])
            t, p = T[order, i], P[order, i]
            if max_points and len(t) > max_points:
                sel = np.linspace(0, len(t) - 1, max_points).astype(int)
                t, p, x = t[sel], p[sel], sel
            else:
                x = np.arange(len(t))
            ax.plot(x, t, lw=1.6, color='k', label='raycaster (truth)', zorder=3)
            ax.scatter(x, p, s=4, alpha=0.35, color='tab:red',
                       label='FFNN (prediction)', zorder=2)
            ax.set_title(f"{names[i]}   R2 = {r2_score(T[:, i], P[:, i]):.4f}",
                         fontsize=10)
            ax.set_xlabel('sample rank (by true value)')
            ax.set_ylabel(f"{names[i]} [{_unit_of(names[i])}]")
            ax.ticklabel_format(style='sci', scilimits=(0, 0), axis='y')
            ax.grid(alpha=0.3)
            if i == 0:
                ax.legend(fontsize=8, loc='upper left')
        for j in range(n_out, len(axes)):
            axes[j].axis('off')
        _finish(fig, 'pred_vs_true_sorted')

    if index_view:
        w = n_samples if not max_points else min(max_points, n_samples)
        fig, axes = _grid(f"Truth vs prediction - {tag}, dataset order "
                          f"(first {w} samples)")
        x = np.arange(w)
        for i in range(n_out):
            ax = axes[i]
            ax.plot(x, T[:w, i], lw=1.0, color='k', label='raycaster (truth)')
            ax.plot(x, P[:w, i], lw=1.0, color='tab:red', alpha=0.75,
                    label='FFNN (prediction)')
            ax.set_title(names[i], fontsize=10)
            ax.set_xlabel('sample index')
            ax.set_ylabel(f"{names[i]} [{_unit_of(names[i])}]")
            ax.ticklabel_format(style='sci', scilimits=(0, 0), axis='y')
            ax.grid(alpha=0.3)
            if i == 0:
                ax.legend(fontsize=8, loc='upper left')
        for j in range(n_out, len(axes)):
            axes[j].axis('off')
        _finish(fig, 'pred_vs_true_index')

    if parity:
        fig, axes = _grid(f"Parity plots - {tag}")
        for i in range(n_out):
            ax = axes[i]
            t, p = T[:, i], P[:, i]
            if max_points and len(t) > max_points:
                sel = np.random.default_rng(0).choice(len(t), max_points, False)
                ts, ps = t[sel], p[sel]
            else:
                ts, ps = t, p
            ax.scatter(ts, ps, s=4, alpha=0.3, color='tab:blue')
            lo, hi = min(t.min(), p.min()), max(t.max(), p.max())
            ax.plot([lo, hi], [lo, hi], 'k--', lw=1.2, label='y = x')
            ax.set_title(f"{names[i]}   R2 = {r2_score(t, p):.4f}   "
                         f"RMSE = {float(np.sqrt(((p-t)**2).mean())):.2e}",
                         fontsize=10)
            ax.set_xlabel(f"truth [{_unit_of(names[i])}]")
            ax.set_ylabel(f"prediction [{_unit_of(names[i])}]")
            ax.ticklabel_format(style='sci', scilimits=(0, 0), axis='both')
            ax.grid(alpha=0.3)
            ax.set_aspect('equal', adjustable='box')
            if i == 0:
                ax.legend(fontsize=8, loc='upper left')
        for j in range(n_out, len(axes)):
            axes[j].axis('off')
        _finish(fig, 'parity')

    if residuals:
        fig, axes = _grid(f"Residuals (prediction - truth) - {tag}")
        for i in range(n_out):
            ax = axes[i]
            e = P[:, i] - T[:, i]
            ax.hist(e, bins=80, color='tab:purple', alpha=0.8)
            ax.axvline(0, color='k', lw=1)
            ax.set_title(f"{names[i]}   bias = {e.mean():+.2e}   "
                         f"std = {e.std():.2e}", fontsize=10)
            ax.set_xlabel(f"error [{_unit_of(names[i])}]")
            ax.set_ylabel('count')
            ax.ticklabel_format(style='sci', scilimits=(0, 0), axis='x')
            ax.grid(alpha=0.3)
        for j in range(n_out, len(axes)):
            axes[j].axis('off')
        _finish(fig, 'residuals')

    print(f"\nPer-component summary ({tag}, {n_samples} samples)")
    print(f"{'component':<20} {'R2':>9} {'RMSE':>12} {'MAE':>12} "
          f"{'max|err|':>12} {'bias':>12} {'RMSE/std':>10}")
    print("-" * 92)
    for i in range(n_out):
        e = P[:, i] - T[:, i]
        sd = T[:, i].std()
        print(f"{names[i]:<20} {r2_score(T[:, i], P[:, i]):>9.5f} "
              f"{np.sqrt((e**2).mean()):>12.4e} {np.abs(e).mean():>12.4e} "
              f"{np.abs(e).max():>12.4e} {e.mean():>+12.4e} "
              f"{np.sqrt((e**2).mean())/(sd+1e-30):>10.4f}")
    return figs

In [ ]:
plot_dir = os.path.join(dataset_path, "prediction_plots_noq")

T_val, P_val = collect_predictions(model, val_dataset,
                                   output_scaler=output_scaler, device=device)
print(f"Validation: {T_val.shape[0]} samples x {T_val.shape[1]} outputs")
plot_predictions_vs_targets(T_val, P_val, names=OUTPUT_NAMES,
                            save_dir=plot_dir, tag='val', max_points=None)
plt.show()

T_test, P_test = collect_predictions(model, test_dataset,
                                     output_scaler=output_scaler, device=device)
plot_predictions_vs_targets(T_test, P_test, names=OUTPUT_NAMES,
                            save_dir=plot_dir, tag='test', max_points=None)
plt.show()

# 9. Deployment artifact

`ScaledDragModel` bundles the scalers with the network: raw `[N, 22]` in physical units in, `[N, 6]` in N and
N·m out. The scaling constants are registered buffers, so they follow `.to(device)` and survive
`torch.jit.script`.

In [ ]:
class ScaledDragModel(nn.Module):
    """Scaling, network and de-scaling in one module

        wrench = net((x - in_mean) / in_scale) * out_scale + out_mean

    with an optional log10 on the density column, as the input scaler was fitted. The buffers come from the
    fitted scalers: never re-fit them at evaluation time
    """

    def __init__(self, net, input_scaler, output_scaler):
        super().__init__()
        self.net = net
        self.register_buffer("in_mean", input_scaler.mean.clone())
        self.register_buffer("in_scale", input_scaler.scale.clone())
        self.register_buffer("out_mean", output_scaler.mean.clone())
        self.register_buffer("out_scale", output_scaler.scale.clone())
        self.log_columns = list(input_scaler.log_columns)
        self.log_floor = float(input_scaler.log_floor)

    def forward(self, x_raw: torch.Tensor) -> torch.Tensor:
        x = x_raw
        if len(self.log_columns) > 0:
            x = x.clone()
            for c in self.log_columns:
                x[:, c] = torch.log10(x[:, c].clamp_min(self.log_floor))
        x = (x - self.in_mean) / self.in_scale
        return self.net(x) * self.out_scale + self.out_mean


deploy_model = ScaledDragModel(model, input_scaler, output_scaler).to(device).eval()

# The wrapper must reproduce the manual pipeline
_raw = X_test_raw[:256].to(device)
with torch.no_grad():
    _direct = deploy_model(_raw)
    _manual = output_scaler.inverse_transform(model(input_scaler.transform(_raw)))
_den = _manual.abs().clamp_min(_manual.abs().median().clamp_min(1e-30))
print(f"wrapper vs manual, max abs : {(_direct - _manual).abs().max():.3e}")
print(f"wrapper vs manual, max rel : {((_direct - _manual).abs() / _den).max():.3e}")

_p = os.path.join(dataset_path, "drag_surrogate_deploy_noq.pt")
torch.save(deploy_model, _p)
_s = torch.jit.script(deploy_model)
with torch.no_grad():
    print(f"TorchScript max abs diff   : {(_s(_raw) - _direct).abs().max():.3e}")
_s.save(os.path.join(dataset_path, "drag_surrogate_deploy_noq_scripted.pt"))
print(f"saved {_p} (+ scripted)")

# 10. Evaluation on independently collected data

Evaluates the trained model on a dataset collected separately (different epoch) with the training scalers
reused verbatim. Two things decide whether the comparison is meaningful, and both are printed:

* **The collection seed.** Datasets sharing a seed share their random attitudes and boom configurations, so
  only the orbital conditions would differ.
* **The density.** A different atmosphere model than in training makes any degradation a model mismatch
  rather than a temporal shift. Fewer than ten distinct ρ values means a static model (e.g. USSA76).

In [ ]:
# Final evaluation of the test split and of an independently collected set. For each dataset:
#   1. density report
#   2. per-component metrics, with and without the worst 1%, and RMSE/MAE
#      (far above the Gaussian 1.25 means a few samples drive the score)
#   3. bootstrap confidence intervals of R2
#   4. attitude-band stratification
#   5. LaTeX rows

RHO_ARTIFACT = 1e-30  # below this the density is not physical at any LEO altitude
BANDS = [(0, 30), (30, 60), (60, 90), (90, 120), (120, 150), (150, 180)]

Fm = lambda A: np.linalg.norm(A[:, :3], axis=1)
Tm = lambda A: np.linalg.norm(A[:, 3:], axis=1)


def quat_geodesic_deg(q1, q2):
    """Angle of the relative rotation [deg], invariant to the quaternion sign"""
    d = np.abs((q1 * q2).sum(1)).clip(0.0, 1.0)
    return np.degrees(2.0 * np.arccos(d))


def evaluate_set(Xr, yr, tag, model, input_scaler, output_scaler,
                 rho_col=RHO_COL, drop_anomalous_episodes=True,
                 samples_per_episode=None, n_boot=500):
    """Full report for one raw (Xr, yr) pair"""
    Xr = Xr.detach().cpu().float()
    yr = yr.detach().cpu().float()
    rho = Xr[:, rho_col].numpy()

    print("=" * 78)
    print(f"{tag}: {Xr.shape[0]} samples")
    print("=" * 78)

    # 1. density
    alt = float(Xr[:, 12:15].double().norm(dim=1).mean()) - 6378.137
    n_rho = len(np.unique(rho))
    frac_lo = float((rho < RHO_ARTIFACT).mean())
    print(f"altitude {alt:.1f} km   rho [{rho.min():.3e}, {rho.max():.3e}] "
          f"median {np.median(rho):.3e}")
    print(f"  {n_rho} distinct rho values"
          + ("   <-- STATIC atmosphere model, no epoch dependence" if n_rho < 10 else ""))
    print(f"  {100*frac_lo:.1f}% below {RHO_ARTIFACT:.0e} kg/m^3"
          + ("   <-- density dropout present; relative errors are inflated"
             if frac_lo > 0.01 else "   (clean)"))

    keep = np.ones(len(rho), dtype=bool)

    # Anomalously high density, usually a whole window
    if drop_anomalous_episodes:
        hi = rho > 10 * np.median(rho)
        if hi.any():
            if samples_per_episode:
                ep = np.arange(len(rho)) // samples_per_episode
                bad_ep = np.unique(ep[hi])
                keep &= ~np.isin(ep, bad_ep)
                print(f"  excluding episode(s) {bad_ep.tolist()}: rho exceeds the "
                      f"median by >10x ({int((~keep).sum())} samples)")
            else:
                keep &= ~hi
                print(f"  excluding {int(hi.sum())} samples with rho >10x median")

    # 2. metrics
    ds = TensorDataset(input_scaler.transform(Xr), output_scaler.transform(yr))
    T, P = collect_predictions(model, ds, output_scaler=output_scaler, device=device)
    T, P, Xk, rho_k = T[keep], P[keep], Xr[keep], rho[keep]
    err = P - T
    print(f"  evaluating on {len(T)} samples\n")

    print(f"{'component':<16} {'R2':>9} {'R2 -1%':>9} {'medAE':>11} {'p99 AE':>11} "
          f"{'RMSE/MAE':>9} {'bias/MAE':>9}")
    print("-" * 82)
    per_comp = {}
    for i, nm in enumerate(OUTPUT_NAMES):
        ae = np.abs(err[:, i])
        k1 = ae <= np.percentile(ae, 99)
        per_comp[nm] = dict(
            r2=r2_score(T[:, i], P[:, i]),
            r2_trim=r2_score(T[k1, i], P[k1, i]),
            med_ae=float(np.median(ae)), p99=float(np.percentile(ae, 99)),
            kurt=float(np.sqrt((ae ** 2).mean()) / ae.mean()),
            bias=float(err[:, i].mean() / ae.mean()))
        c = per_comp[nm]
        print(f"{nm:<16} {c['r2']:>9.4f} {c['r2_trim']:>9.4f} {c['med_ae']:>11.3e} "
              f"{c['p99']:>11.3e} {c['kurt']:>9.1f} {c['bias']:>9.2f}")
    print("-" * 82)
    print("RMSE/MAE near 1.25 is Gaussian, above ~5 a few samples dominate the aggregate R2\n")

    # 3. bootstrap confidence intervals
    rng = np.random.default_rng(0)
    idx = rng.integers(0, len(T), (n_boot, len(T)))
    print(f"{'component':<16} {'R2':>9} {'2.5%':>9} {'97.5%':>9}")
    print("-" * 46)
    for i, nm in enumerate(OUTPUT_NAMES):
        bs = np.array([r2_score(T[s, i], P[s, i]) for s in idx])
        lo, hi_ = np.percentile(bs, [2.5, 97.5])
        per_comp[nm]['ci'] = (float(lo), float(hi_))
        print(f"{nm:<16} {per_comp[nm]['r2']:>9.4f} {lo:>9.4f} {hi_:>9.4f}")

    # 4. attitude bands
    ang = quat_geodesic_deg(Xk[:, 0:4].numpy(), Xk[:, 8:12].numpy())
    print(f"\n{'band':>10} {'n':>7} {'R2 |F|':>8} {'med rel':>9} {'med signed':>11} "
          f"{'peak ratio':>11} {'rel |tau|':>10}")
    print("-" * 70)
    rows = []
    for lo, hi_ in BANDS:
        m = (ang >= lo) & (ang < hi_ if hi_ < 180 else ang <= 180)
        if m.sum() < 50:
            continue
        tf = np.maximum(Fm(T[m]), 1e-30); pf = Fm(P[m])
        tt = np.maximum(Tm(T[m]), 1e-30); pt = Tm(P[m])
        top = tf >= np.percentile(tf, 90)
        r = dict(band=f"{lo}--{hi_}", n=int(m.sum()),
                 r2=r2_score(tf, pf),
                 med_rel=float(np.median(np.abs(pf - tf) / tf)),
                 med_sgn=float(np.median((pf - tf) / tf)),
                 peak=float(np.median(pf[top] / tf[top])),
                 rel_tau=float(np.median(np.abs(pt - tt) / tt)))
        rows.append(r)
        print(f"{r['band']:>10} {r['n']:>7} {r['r2']:>8.4f} {r['med_rel']:>9.3f} "
              f"{r['med_sgn']:>+11.3f} {r['peak']:>11.3f} {r['rel_tau']:>10.3f}")

    # 5. LaTeX
    print(f"\n% --- LaTeX rows for {tag} ---")
    for r in rows:
        n = f"{r['n']:,}".replace(",", "\\,")
        print(f"{r['band']:<9}& {n:>7} & {r['r2']:.3f} & {r['med_rel']:.3f} & "
              f"${r['med_sgn']:+.3f}$ & {r['peak']:.3f} & {r['rel_tau']:.3f} \\\\")

    return dict(T=T, P=P, X=Xk, rho=rho_k, ang=ang, per_component=per_comp,
                bands=rows, kept=int(keep.sum()), total=int(len(keep)))


# Test split
res_test = evaluate_set(X_test_raw, y_test_raw, "TEST SPLIT", model,
                        input_scaler, output_scaler, rho_col=RHO_COL,
                        drop_anomalous_episodes=False)

# Independently collected set (six months later)
OOD_PATH = "datasets/episodes_ood/"
files = sorted(glob(os.path.join(OOD_PATH, "episode_*.pt")))
if files:
    _i, _o, _seeds, _n = [], [], set(), []
    for f in files:
        d = torch.load(f, weights_only=False)
        _i.append(d['inputs']); _o.append(d['outputs'])
        _seeds.add(d.get('metadata', {}).get('seed', None))
        _n.append(len(d['inputs']))
    Xo, yo = torch.cat(_i, 0).float(), torch.cat(_o, 0).float()
    ok = torch.isfinite(Xo).all(1) & torch.isfinite(yo).all(1)
    Xo, yo = Xo[ok], yo[ok]

    spe = _n[0] if len(set(_n)) == 1 else None
    print(f"\ncollection seed(s): {_seeds}"
          + ("   <-- SAME AS TRAINING: geometry is not unseen" if _seeds == {seed}
             else "   (differs from training -> unseen attitudes and boom configs)"))

    res_ood = evaluate_set(Xo, yo, "OOD MONTH 6", model, input_scaler,
                           output_scaler, rho_col=RHO_COL,
                           drop_anomalous_episodes=True, samples_per_episode=spe)
else:
    print(f"no episode files under {OOD_PATH}")

# 11. Tables and figures

Per-component error tables (with LaTeX rows) and the residual figure for the test split and the independent set.

In [ ]:
# Per-component error table: MAE and RMSE, plus MAE/sigma and RMSE/sigma, which are dimensionless and
# comparable between forces and torques
def component_table(T, P, names=None, label="", latex=True):
    names = names or OUTPUT_NAMES
    err = P - T
    rows = []
    print("=" * 104)
    print(f"PER-COMPONENT ERRORS -- {label}   (n = {len(T)})")
    print("=" * 104)
    print(f"{'component':<14} {'R2':>8} {'MAE':>12} {'RMSE':>12} {'medAE':>12} "
          f"{'p99 AE':>12} {'sigma(T)':>12} {'MAE/sig':>8} {'RMSE/MAE':>9}")
    print("-" * 104)
    for i, nm in enumerate(names):
        e = err[:, i]
        ae = np.abs(e)
        mae = float(ae.mean())
        rmse = float(np.sqrt((e ** 2).mean()))
        sd = float(T[:, i].std())
        r = dict(name=nm, r2=float(r2_score(T[:, i], P[:, i])), mae=mae,
                 rmse=rmse, med=float(np.median(ae)),
                 p99=float(np.percentile(ae, 99)), sigma=sd,
                 mae_n=mae / (sd + 1e-30), rmse_n=rmse / (sd + 1e-30),
                 ratio=rmse / (mae + 1e-30), bias=float(e.mean()))
        rows.append(r)
        print(f"{nm:<14} {r['r2']:>8.4f} {mae:>12.4e} {rmse:>12.4e} "
              f"{r['med']:>12.4e} {r['p99']:>12.4e} {sd:>12.4e} "
              f"{r['mae_n']:>8.4f} {r['ratio']:>9.2f}")
    print("-" * 104)
    print(f"mean R2 = {np.mean([r['r2'] for r in rows]):.4f}   "
          f"force {np.mean([r['r2'] for r in rows[:3]]):.4f}   "
          f"torque {np.mean([r['r2'] for r in rows[3:]]):.4f}")
    print(f"RMSE/MAE range = [{min(r['ratio'] for r in rows):.2f}, "
          f"{max(r['ratio'] for r in rows):.2f}]   (Gaussian = 1.253)")

    if latex:
        print(f"\n% --- LaTeX rows, {label} ---")
        sym = ["$F_x$", "$F_y$", "$F_z$", r"$\tau_x$", r"$\tau_y$", r"$\tau_z$"]
        for s, r in zip(sym, rows):
            print(f"{s:<9}& {r['r2']:.3f} & {r['mae_n']:.4f} & {r['rmse_n']:.4f} "
                  f"& {r['ratio']:.2f} \\\\")
    return rows


rows_test = component_table(res_test['T'], res_test['P'],
                            label="TEST SPLIT")
rows_ood = component_table(res_ood['T'], res_ood['P'],
                           label="SIX MONTHS LATER")


SYMBOLS = {'net_force[0]': 'F_x', 'net_force[1]': 'F_y', 'net_force[2]': 'F_z',
           'net_torque[0]': 'tau_x', 'net_torque[1]': 'tau_y', 'net_torque[2]': 'tau_z'}


def plot_residuals(T, P, names=None, save_path=None, tag="test",
                   normalise=True, clip_pct=99.5, logy=False, bins=60):
    """Residual histograms with a matched Gaussian overlay

    normalise: divide each residual by the target sigma of its component, so all panels share the axis
    clip_pct: percentile of the x-range, the excluded fraction is annotated in each panel
    """
    names = names or OUTPUT_NAMES
    n = T.shape[1]
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), dpi=140)
    axes = np.atleast_1d(axes).ravel()

    for i in range(n):
        ax = axes[i]
        e = P[:, i] - T[:, i]
        sd_t = T[:, i].std()
        if normalise:
            e = e / (sd_t + 1e-30)
            unit = r"error / $\sigma$"
        else:
            unit = f"error [{'N·m' if 'torque' in names[i].lower() else 'N'}]"

        lim = np.percentile(np.abs(e), clip_pct)
        outside = float((np.abs(e) > lim).mean())
        m = np.abs(e) <= lim

        ax.hist(e[m], bins=bins, color='tab:purple', alpha=0.75, density=True,
                label='residuals')
        xs = np.linspace(-lim, lim, 400)
        mu, sg = e.mean(), e.std()
        ax.plot(xs, np.exp(-0.5 * ((xs - mu) / sg) ** 2) / (sg * np.sqrt(2 * np.pi)),
                'k--', lw=1.3, label=r'Gaussian, matched $\mu,\sigma$')
        ax.axvline(0, color='k', lw=0.8)
        if logy:
            ax.set_yscale('log')

        symbol = SYMBOLS.get(names[i], names[i])
        ratio = np.sqrt((e**2).mean()) / np.abs(e).mean()
        print(f"Component {symbol}: Bias = {mu:+.2e}, RMSE/MAE = {ratio:.2f}")

        ax.set_title(f"{symbol}   bias = {mu:+.2e}   RMSE/MAE = {ratio:.2f}", fontsize=9.5)
        ax.set_xlabel(unit, fontsize=9)
        ax.set_ylabel('normalised frequency', fontsize=9)
        ax.set_xlim(-lim, lim)
        ax.grid(alpha=0.25)
        ax.tick_params(labelsize=8)
        if i == 0:
            ax.legend(fontsize=7.5, loc='upper left')
        ax.text(0.98, 0.95, f"{100*outside:.1f}% beyond axis",
                transform=ax.transAxes, ha='right', va='top', fontsize=7,
                color='0.35')

    for j in range(n, len(axes)):
        axes[j].axis('off')
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path:
        fig.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"saved {save_path}")
    return fig


plot_residuals(res_test['T'], res_test['P'], tag="test set",
               save_path=os.path.join(dataset_path, "residuals_test.png"))
plt.show()